# Browse cached batch results

Load every batch stored in `cache.db` (written by `explore.ipynb` / `process_batch`) and print the prompts and responses for the most recent runs.


## Load cache


In [ ]:
import json
import sqlite3
from pathlib import Path


def repo_root() -> Path:
    """Walk upward from cwd until pyproject.toml is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (no pyproject.toml).")


DB_PATH = repo_root() / "cache.db"
DB_PATH


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='batch_cache'"
).fetchall()
if not tables:
    print("No cache yet — run explore.ipynb first to generate responses.")
    rows = []
else:
    rows = conn.execute(
        "SELECT key, params_json, responses_json, created_at FROM batch_cache ORDER BY created_at"
    ).fetchall()
conn.close()

batches = []
for key, params_json, responses_json, created_at in rows:
    params = json.loads(params_json)
    responses = json.loads(responses_json)
    messages_list = params.get("messages_list", [])
    if not messages_list:
        continue
    messages = messages_list[0]
    user_prompt = next((m["content"] for m in messages if m["role"] == "user"), "")
    system_prompt = next((m["content"] for m in messages if m["role"] == "system"), "N/A")
    batches.append({
        "key": key,
        "created_at": created_at,
        "model": params.get("model", "unknown"),
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "responses": [
            r.get("choices", [{}])[0].get("message", {}).get("content", "")
            for r in responses
        ],
        "n": len(responses),
    })

print(f"Found {len(batches)} cached batches")


## Latest batch prompts

Set `N_RECENT` to control how many of the most recent batches to show.


In [ ]:
N_RECENT = 2

for i, batch in enumerate(batches[-N_RECENT:]):
    print(f"{'=' * 80}")
    print(f"Batch {i + 1} | {batch['created_at']} | {batch['model']} | {batch['n']} responses")
    print(f"\nSystem prompt:\n{batch['system_prompt']}\n")
    print(f"User prompt:\n{batch['user_prompt']}")
    print(f"{'=' * 80}\n")


## Latest batch responses


In [ ]:
for i, batch in enumerate(batches[-N_RECENT:]):
    print(f"Batch {i + 1} responses:")
    for j, resp in enumerate(batch["responses"]):
        print(f"\n--- Response {j + 1} ---")
        print(resp)
    print()
